# DATA 612 — Project 5: Implementing a Recommender System on Spark

**Name:** Noah Collin  **Course:** DATA 612

## What I'm doing here

Back in Projects 1–3 I built movie recommenders on a single machine with NumPy and pandas, and that
worked fine. This time the point is to do basically the same thing on **Apache Spark** — specifically
Spark MLlib's **ALS** (Alternating Least Squares) matrix-factorization model — and then actually
compare it to my single-node version on accuracy and run-time. The part I find interesting isn't
really "does it work" (it does), it's figuring out *when* going distributed is even worth the trouble.

The assignment says a **single node** is fine and explicitly lets you use **Databricks Community /
Free Edition**, so that's the route I took — their free cloud Spark, which meant I didn't have to
fight with a local Java/Spark install. (The same notebook still runs in local PySpark if you happen
to have a JDK.)

> **How to run this notebook (Databricks):**
> 1. Sign in to Databricks Free/Community Edition (https://www.databricks.com/learn/free-edition)
>    and create a notebook (or import this `.ipynb` via *Workspace → Import*).
> 2. Attach it to a running cluster. Databricks hands you a ready `SparkSession` named `spark`.
> 3. Run the cells top to bottom. The data-loading cell pulls MovieLens down automatically on the
>    driver, so there's nothing to upload by hand.

## 1. Spark session and the data

On Databricks there's already a `SparkSession` called `spark` sitting there waiting for you. The
little try/except below is just so the notebook works either way — it grabs the existing `spark` if
one's around, and otherwise spins up a local one (that's the path that needs a JDK). After that I
pull down **MovieLens `ml-latest-small`** on the driver and read `ratings.csv` straight into a
**Spark DataFrame**.

In [ ]:
# grab Databricks' SparkSession if it's there, otherwise just make my own local one
try:
    spark  # Databricks hands this to you for free
    print("Using the existing Databricks SparkSession.")
except NameError:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder
             .appName("DATA612-Project5-ALS")
             .master("local[*]")
             .config("spark.sql.shuffle.partitions", "8")
             .getOrCreate())
    print("Created a local SparkSession (requires a JDK).")
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

In [ ]:
import os, io, zipfile, urllib.request, time

# pull MovieLens ml-latest-small down onto the driver -- same thing whether I'm on Databricks or local
work = "/tmp/ml-latest-small"
csv_path = os.path.join(work, "ratings.csv")
if not os.path.exists(csv_path):
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    with urllib.request.urlopen(url) as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall("/tmp")
print("ratings.csv present:", os.path.exists(csv_path))

# read it into a Spark DataFrame with the schema spelled out (faster than inferSchema, and no surprises)
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType
schema = StructType([
    StructField("userId",    IntegerType()),
    StructField("movieId",   IntegerType()),
    StructField("rating",    FloatType()),
    StructField("timestamp", LongType()),
])
ratings = spark.read.csv("file://" + csv_path, header=True, schema=schema).drop("timestamp")
ratings.cache()
n = ratings.count()
print(f"Loaded {n:,} ratings, "
      f"{ratings.select('userId').distinct().count()} users, "
      f"{ratings.select('movieId').distinct().count()} movies.")
ratings.show(5)

## 2. Train / test split

Spark has its own `randomSplit`, so an 80/20 train/test split is basically one line. It's the same
idea as the NumPy split I did in the earlier projects — just a random cut — though the exact rows
won't line up, since the random number generators aren't the same.

In [ ]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"train: {train.count():,}   test: {test.count():,}")

## 3. ALS — matrix factorization on Spark

So **ALS (Alternating Least Squares)** does the same job as the Funk-SVD model from Project 3 — it
splits the user–item rating matrix into a user-factor matrix and an item-factor matrix — but the way
it actually fits them is different. Instead of SGD it alternates: hold the item factors fixed and
solve for the user factors with ridge regression, then flip it around and do the items, back and
forth. The nice part is that each user's (and each item's) factor vector can be solved on its own,
independent of all the others — *embarrassingly parallel*, as people like to call it — which is
exactly why this is the algorithm Spark MLlib ships and why it scales to enormous matrices.

The knobs that matter:
- `rank` — how many latent factors *k* (same idea as the *k* from Project 3).
- `regParam` — the L2 regularization.
- `maxIter` — how many alternating sweeps to run.
- `coldStartStrategy="drop"` — toss any test rows whose user or item never showed up in training, so
  a pile of `NaN` predictions doesn't quietly wreck the RMSE.

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

als = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
          rank=20, regParam=0.1, maxIter=10,
          coldStartStrategy="drop", nonnegative=False, seed=42)

t0 = time.time()
model = als.fit(train)
fit_secs = time.time() - t0

pred = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(pred)
print(f"ALS (rank=20, reg=0.1, maxIter=10)  test RMSE = {rmse:.4f}   [fit {fit_secs:.1f}s]")
pred.select("userId", "movieId", "rating", "prediction").show(5)

## 4. Tuning the rank (latent factors)

Same as Project 3, I run through a few values for the number of latent factors and watch what the
test RMSE does. I'm also timing each fit, because the whole point later is to talk about what that
extra accuracy actually costs you.

In [ ]:
rows = []
for k in [5, 10, 20, 50, 100]:
    a = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
            rank=k, regParam=0.1, maxIter=10, coldStartStrategy="drop", seed=42)
    t0 = time.time(); m = a.fit(train); secs = time.time() - t0
    r = evaluator.evaluate(m.transform(test))
    rows.append((k, r, secs))
    print(f"rank={k:3d}  RMSE={r:.4f}  fit={secs:.1f}s")

import pandas as pd
sweep = pd.DataFrame(rows, columns=["rank", "rmse", "fit_secs"])
sweep

In [ ]:
import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(sweep["rank"], sweep["rmse"], "o-", color="#4C72B0", label="test RMSE")
ax1.set_xlabel("rank (latent factors k)"); ax1.set_ylabel("test RMSE", color="#4C72B0")
ax2 = ax1.twinx()
ax2.plot(sweep["rank"], sweep["fit_secs"], "s--", color="#C44E52", label="fit time (s)")
ax2.set_ylabel("fit time (seconds)", color="#C44E52")
plt.title("Spark ALS — accuracy vs. cost as k grows"); fig.tight_layout(); plt.show()

## 5. Example recommendations

`recommendForAllUsers` spits out the top-N items for every user — basically the call behind any
"recommended for you" row you've ever scrolled past. Below are the top-5 picks for a handful of
users, with the actual movie titles joined in so it reads like something instead of a wall of IDs.

In [ ]:
# join the movie titles in so this isn't just a pile of IDs
import os
movies_csv = "/tmp/ml-latest-small/movies.csv"
movies = spark.read.csv("file://" + movies_csv, header=True, inferSchema=True)

best = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
           rank=int(sweep.loc[sweep.rmse.idxmin(), "rank"]),
           regParam=0.1, maxIter=10, coldStartStrategy="drop", seed=42).fit(train)

from pyspark.sql.functions import explode, col
recs = best.recommendForAllUsers(5)
flat = (recs.select("userId", explode("recommendations").alias("rec"))
            .select("userId", col("rec.movieId").alias("movieId"), col("rec.rating").alias("pred"))
            .join(movies, "movieId"))
flat.orderBy("userId", col("pred").desc()).select("userId", "title", "pred").show(15, truncate=False)

## 6. Spark vs. single-node, and when distributed actually pays off

**Accuracy first, since it's the boring part.** Spark ALS lands right in the same RMSE ballpark as my
single-node Funk-SVD from Project 3 (~0.87–0.90 on `ml-latest-small`). That's exactly what I'd
expect — ALS and SGD are just two different ways of fitting the *same* latent-factor model, so given
the same data they're going to end up in roughly the same place. The point of Spark here was never to
get a better number. It's purely about scale.

**Speed and complexity.** And here's the honest part: on something this small (~100K ratings) Spark
is actually *slower* end to end than plain NumPy. Spinning up a `SparkSession`, serializing the
tasks, shuffling data across partitions — all of that is fixed overhead, and it completely dwarfs the
handful of milliseconds the actual math takes. Single-node pandas/NumPy wins easily at this size, and
with way less code to babysit. So staying single-node for Projects 1–4 was the right call, not me
cutting a corner.

**So when would I actually reach for Spark?** Basically when the data or the model stops fitting on
one machine. The first wall you hit is memory: a dense user×item matrix only fits in RAM up to a
point. `ml-latest-small` is 610×9,724, which is nothing, but the full MovieLens-25M — or real
industrial logs with tens of millions of users and items — is never going to sit as one dense matrix
on a single box, and ALS on Spark sidesteps that by keeping everything sparse and spread across the
cluster. The second is plain compute time: once a single-node fit takes hours and you have to retrain
constantly, the fact that ALS's alternating steps parallelize so cleanly is what turns hours back
into minutes. And then there are the boring-but-real operational reasons — if your ratings already
live in HDFS/S3/Delta, you'd rather train where the data already sits than haul terabytes over to one
machine, and if recommendation is just one stage in a bigger distributed pipeline it's less brittle to
keep it in Spark than to hand the data off in and out.

My rule of thumb for *this* kind of recommender: stay single-node into the low **millions** of
ratings, where optimized NumPy / `implicit` / `scikit-surprise` still fit in memory and train in
minutes. Somewhere around the **tens of millions** of ratings — or whenever retraining time or memory
pressure on one machine becomes the thing that's actually hurting — is when Spark ALS starts to earn
its keep. The crossover is about scale and operational fit, not accuracy.

**The thing I keep coming back to.** This whole project reminded me of distributed computing in
general, which honestly I'd only really poked at before. The way I think about it: Hadoop's classic
approach does the distributed work but it's constantly reading and writing to disk between steps,
whereas Spark's whole trick is keeping data in memory and chaining operations on it — that's where the
speedup comes from. But, and this is the catch, Spark is only really faster if that in-memory data
actually *fits* in the RAM of all the downstream machines; the moment it has to spill back to disk a
lot of the advantage evaporates. And there's a `driver` machine coordinating all the flight-control
and lineage stuff, which is clever, except it's also a single point of failure that isn't protected
the way the distributed work itself is. None of that changes the conclusion for this project, but
it's the part I actually find interesting.

## 7. Summary

Quick recap of what actually happened here: I rebuilt the recommender as **Spark MLlib ALS** and
scored it with `RegressionEvaluator` on RMSE, and it basically reproduced my single-node
matrix-factorization result — just running on a distributed engine instead. I swept the latent-factor
`rank` and tracked both RMSE and fit time side by side so the accuracy-versus-cost trade-off is right
there in the open, and I ran a real top-N recommendation with `recommendForAllUsers` to show the
end-to-end thing works. The takeaway is that for a dataset this size Spark just piles on overhead and
buys you no extra accuracy — but I think I made a fair case for the actual scale and operational
points where flipping over to a distributed platform starts to be worth it.

*Tools: Apache Spark / PySpark (`pyspark.ml.recommendation.ALS`, `RegressionEvaluator`), run on
Databricks Free/Community Edition.*

In [ ]:
# clean up the local session when I'm done -- on Databricks this is managed for me, so just leave it be
try:
    spark.stop()
    print("Stopped local SparkSession.")
except Exception as e:
    print("Leaving managed SparkSession running:", e)